# Role C (evaluation): measure OCR and the diacritic corrector

Needs the Colab secret `GITHUB_TOKEN` with **Notebook access** switched on for this notebook.

**First:** choose File > Save a copy in Drive and work in the copy, so the notebook in the shared folder stays clean. In the copy, switch on Notebook access for `GITHUB_TOKEN` again. Branch: `c/demo` (if it does not exist on GitHub yet, set `BRANCH = "main"` in cell 1).

What this notebook does, in order:

1. Connect to Drive and GitHub.
2. Settings that must be decided **before** looking at numbers.
3. Load the data and check it (coverage of the val split, labels, NFC, passages).
4. Raw OCR quality: CER and WER, with the raw text and with whitespace normalized, side by side.
5. Error classes and the hypothesis gate (diacritic share and benefit ceiling).
6. The corrector (a FAKE one is built in for pipeline tests) and the contract check.
7. Fixed / broken table, counted two ways: Option 1 (lines with the same syllable count) and Option 2 (word alignment).
8. Sweep of the confidence threshold tau on val, with two caps on the broken rate (1% and 2%).
9. Bootstrap by passage (main) and by page (val only, for comparison).
10. Score the test split. Off by default, runs once.
11. Save the val results to Drive.

Rules for reading the output:
- Numbers computed from a **draft OCR file** or from the **FAKE corrector** only check that the pipeline runs. They are not results. Output files get a `_DRAFT` and/or `_FAKE` suffix, and the test cell refuses to run while either is active.
- The gate thresholds in cell 2 are decided first. Until they are set, cell 5 does not show the diacritic share.
- Tau is chosen on val only. The test split is scored once, after tau is locked.

## 1. Connect to Drive and GitHub

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")

BRANCH = "c/demo"  # if this branch does not exist on GitHub yet, use "main"

token = userdata.get("GITHUB_TOKEN").strip()
!rm -rf /content/repo
!git clone -q https://{token}@github.com/wickyhien18/Vietnamese-Handwriting-OCR.git /content/repo
!git -C /content/repo checkout -q {BRANCH}
%cd /content/repo
!pip install -q -r requirements.txt

import sys

REPO_ROOT = "/content/repo"
sys.path.insert(0, REPO_ROOT)

DRIVE_ROOT = "/content/drive/MyDrive/Vietnamese-Handwriting-OCR"
DATA_DIR = f"{DRIVE_ROOT}/data"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
OUT_DIR = f"{DRIVE_ROOT}/outputs"

## 2. Settings decided before looking at numbers

Fill in the gate thresholds **before** the diacritic share is shown. Cell 5 stays hidden while they are `None`.

In [ ]:
OCR_OUT_PATH = f"{OUT_DIR}/ocr_out.jsonl"

# True while ocr_out.jsonl is a draft (for example a short training run).
# Set to False only when the real OCR output is in place. It adds "_DRAFT" to output names.
OCR_IS_DRAFT = True

# --- Hypothesis gate (None = not decided yet) ---
# P_MIN: minimum share of diacritic errors among all errors, for example 0.5
# CEILING_MIN: minimum benefit ceiling (CER * share), for example 0.03
P_MIN = None
CEILING_MIN = None
SMOKE_LINES = 20  # lines used for the hidden smoke test while the gate is not set

# --- Corrector ---
USE_FAKE = True  # True = built-in FAKE corrector (pipeline tests only)
# To use role B's real corrector: set USE_FAKE = False and import it here, for example
#   from src.corrector.<module> import correct as REAL_CORRECT_FN
REAL_CORRECT_FN = None

# --- Choosing tau on val ---
CAPS = [0.01, 0.02]  # caps on the broken rate, compared side by side
FINAL_CAP = None  # after comparing both caps on val, set this to 0.01 or 0.02
TAU_GRID = [round(step * 0.05, 2) for step in range(21)]

# --- Bootstrap ---
N_BOOT = 1000
SEED = 42

# --- Test split (scored once) ---
RUN_TEST = False

## 3. Load the data and check it

The loader finds `contract_checks.py` and `classify_errors.py` anywhere in the repo, so a different folder layout does not break this cell. Any failed check stops the notebook with a message.

In [ ]:
import collections
import glob
import importlib.util
import json
import os
import random
import re
import unicodedata

from src.common.text_utils import normalize_nfc, strip_diacritics
from src.eval.align import DEL, INS, MATCH, SUB, align, corpus_error_rate, edit_distance


def load_repo_module(file_name):
    hits = sorted(
        path for path in glob.glob(f"{REPO_ROOT}/**/{file_name}", recursive=True)
        if "/.git/" not in path
    )
    assert hits, f"{file_name} was not found under {REPO_ROOT}"
    spec = importlib.util.spec_from_file_location(file_name[:-3], hits[0])
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    print("loaded", hits[0])
    return module


def read_jsonl(path):
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def read_json(path):
    with open(path, encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=1)
    print("wrote", path)


contract_checks = load_repo_module("contract_checks.py")
classify_errors = load_repo_module("classify_errors.py")

index_records = read_jsonl(f"{OUT_DIR}/dataset_index.jsonl")
splits = read_json(f"{OUT_DIR}/splits.json")
passages = read_json(f"{OUT_DIR}/passages.json")
ocr_rows = read_jsonl(OCR_OUT_PATH)

problems = contract_checks.check_dataset_index_records(index_records)
problems += contract_checks.check_splits(splits, [rec["id"] for rec in index_records])
assert not problems, "\n".join(problems[:10])
print("index and splits pass the contract checks")

gt_by_id = {rec["id"]: rec["gt_text"] for rec in index_records}
split_sizes = collections.Counter(splits.values())
print("split sizes:", dict(split_sizes))

PAGE_PATTERN = re.compile(r"(?:train_data|test_data)/(\d+)/")
page_to_passage = {}
for passage_id, info in passages["passages"].items():
    for page in info["pages"]:
        if page not in info["excluded_pages"]:
            page_to_passage[page] = passage_id
passage_role = {passage_id: info["role"] for passage_id, info in passages["passages"].items()}


def build_lines(rows, split_name):
    """Turn the ocr_out rows of one split into line records, after strict checks."""
    expected_ids = {rec_id for rec_id, name in splits.items() if name == split_name}
    rows = [row for row in rows if row["split"] == split_name]
    ids = [row["id"] for row in rows]
    assert len(ids) == len(set(ids)), "duplicate ids in ocr_out"
    missing = sorted(expected_ids - set(ids))
    extra = sorted(set(ids) - expected_ids)
    assert not missing and not extra, (
        f"coverage mismatch for {split_name}: {len(missing)} missing {missing[:3]}, "
        f"{len(extra)} extra {extra[:3]}"
    )
    lines = []
    for row in rows:
        gt = gt_by_id[row["id"]]
        assert row["gt_text"] == gt, f"gt_text differs from the index for {row['id']}"
        ocr_raw = row["ocr_text"]
        assert ocr_raw == unicodedata.normalize("NFC", ocr_raw), f"ocr_text is not NFC for {row['id']}"
        match = PAGE_PATTERN.search(row["image_path"])
        assert match, f"cannot read the page number from {row['image_path']}"
        page = match.group(1)
        passage = page_to_passage.get(page)
        assert passage is not None, f"page {page} is not in any passage (excluded page?)"
        assert passage_role[passage] == split_name, f"page {page} belongs to a {passage_role[passage]} passage"
        lines.append({
            "id": row["id"],
            "gt": gt,
            "ocr_raw": ocr_raw,
            "ocr_norm": " ".join(ocr_raw.split()),
            "page": page,
            "passage": passage,
        })
    return lines


val_lines = build_lines(ocr_rows, "val")
print("val lines:", len(val_lines))
print("val passages:", len({line["passage"] for line in val_lines}), "| val pages:", len({line["page"] for line in val_lines}))

# line counts per passage must match passages.json (val passages have no excluded pages)
per_passage = collections.Counter(line["passage"] for line in val_lines)
for passage_id, count in sorted(per_passage.items()):
    expected = passages["passages"][passage_id]["lines"]
    assert count == expected, f"{passage_id}: {count} lines in ocr_out, {expected} in passages.json"
print("line counts per passage match passages.json")

irregular = sum(1 for line in val_lines if line["ocr_raw"] != line["ocr_norm"])
print("lines whose ocr_text has irregular whitespace: %d of %d (%.1f%%)" % (
    irregular, len(val_lines), 100.0 * irregular / len(val_lines)))

## 4. Raw OCR quality (CER and WER)

CER and WER are total edits divided by total reference length (not an average of per-line rates). The "normalized" columns collapse repeated spaces and trim the ends of the OCR text. The gap between the two (raw minus normalized) shows how much whitespace matters. It can be negative, because extra spaces in the OCR text sometimes line up with missing letters.

In [ ]:
_edit_cache = {}


def edits(ref, hyp):
    key = (ref, hyp)
    if key not in _edit_cache:
        _edit_cache[key] = edit_distance(ref, hyp)
    return _edit_cache[key]


def rate(pairs):
    """Pooled error rate: same definition as corpus_error_rate, with a cache."""
    total_ref = sum(len(ref) for ref, _ in pairs)
    return sum(edits(ref, hyp) for ref, hyp in pairs) / total_ref if total_ref else 0.0


def cer_wer(pairs):
    cer = rate(pairs)
    wer = rate([(tuple(ref.split()), tuple(hyp.split())) for ref, hyp in pairs])
    return cer, wer


pairs_raw = [(line["gt"], line["ocr_raw"]) for line in val_lines]
pairs_norm = [(line["gt"], line["ocr_norm"]) for line in val_lines]
assert abs(rate(pairs_raw) - corpus_error_rate(pairs_raw)) < 1e-12

cer_raw, wer_raw = cer_wer(pairs_raw)
cer_norm, wer_norm = cer_wer(pairs_norm)
print("%-22s %10s %10s" % ("", "CER", "WER"))
print("%-22s %9.2f%% %9.2f%%" % ("raw ocr_text", 100 * cer_raw, 100 * wer_raw))
print("%-22s %9.2f%% %9.2f%%" % ("whitespace normalized", 100 * cer_norm, 100 * wer_norm))
print("CER raw minus normalized: %.2f points (can be negative)" % (100 * (cer_raw - cer_norm)))

empty_lines = sum(1 for line in val_lines if not line["ocr_norm"])
same_count = sum(1 for line in val_lines if len(line["gt"].split()) == len(line["ocr_norm"].split()))
total_chars_gt = sum(len(line["gt"]) for line in val_lines)
total_chars_ocr = sum(len(line["ocr_norm"]) for line in val_lines)
print("empty OCR lines:", empty_lines, "of", len(val_lines))
print("lines with the same syllable count as the label: %d of %d (%.1f%%)" % (
    same_count, len(val_lines), 100.0 * same_count / len(val_lines)))
print("characters: label %d, OCR %d" % (total_chars_gt, total_chars_ocr))

per_line_cer = sorted(
    val_lines, key=lambda line: -edits(line["gt"], line["ocr_norm"]) / max(len(line["gt"]), 1))
print("\nfirst 3 lines:")
for line in val_lines[:3]:
    print("  GT :", line["gt"])
    print("  OCR:", repr(line["ocr_raw"]))
print("\n3 worst lines:")
for line in per_line_cer[:3]:
    print("  GT :", line["gt"])
    print("  OCR:", repr(line["ocr_raw"]))

## 5. Error classes and the hypothesis gate

`classify_errors.py` labels every character-level edit as `tone_mark`, `shape_mark`, `tone_and_shape`, `base_letter` or `spacing`. The diacritic share is (tone + shape + both) divided by **all** errors, spacing included. Benefit ceiling = CER x share: the most that a perfect diacritic corrector could remove.

While `P_MIN` or `CEILING_MIN` is `None` this cell only runs a hidden smoke test and prints no numbers.

In [ ]:
def classify_all(pairs):
    total = collections.Counter()
    for ref, hyp in pairs:
        total.update(classify_errors.classify_pair(ref, hyp))
    return total


GATE_READY = P_MIN is not None and CEILING_MIN is not None
gate_report = "not evaluated (thresholds not set)"

if not GATE_READY:
    classify_all(pairs_norm[:SMOKE_LINES])
    print("smoke test ok on %d lines. Numbers stay hidden until P_MIN and CEILING_MIN are set in cell 2." % SMOKE_LINES)
else:
    gate_report = {}
    for name, pairs, cer in (("raw", pairs_raw, cer_raw), ("normalized", pairs_norm, cer_norm)):
        counts = classify_all(pairs)
        share = classify_errors.diacritic_share(counts)
        ceiling = cer * share
        passed = share >= P_MIN and ceiling >= CEILING_MIN
        gate_report[name] = {
            "counts": dict(counts), "diacritic_share": share, "ceiling": ceiling, "passed": passed,
        }
        print("--- %s ---" % name)
        for label in classify_errors.ALL_LABELS:
            print("  %-16s %6d" % (label, counts[label]))
        print("  diacritic share: %.1f%% (minimum %.1f%%)" % (100 * share, 100 * P_MIN))
        print("  benefit ceiling: %.2f points (minimum %.2f)" % (100 * ceiling, 100 * CEILING_MIN))
        print("  gate:", "PASS" if passed else "FAIL")
    if gate_report["raw"]["passed"] != gate_report["normalized"]["passed"]:
        print("\nThe two versions disagree. Decide which one the gate uses and write it in the report.")

## 6. The corrector

`correct(text)` must return `{"text": str, "conf": [float]}` with the same number of syllables, only diacritics changed, and one confidence per syllable. Every call is checked with `check_correct_output`.

The corrector always receives the **whitespace-normalized** OCR text, so any change comes from diacritics only.

**The built-in FAKE corrector peeks at the label of the current line** to produce a mix of fixed, kept and broken syllables. It exists only to test the pipeline. Anything computed with it is saved with a `_FAKE` suffix.

In [ ]:
TONE_MARKS = ["\u0300", "\u0301", "\u0303", "\u0309", "\u0323"]
_fake_state = {"gt": None}


def change_tone(syllable, rng):
    """Put a different tone mark on a syllable (used by the FAKE corrector to break a syllable)."""
    decomposed = unicodedata.normalize("NFD", syllable)
    present = {ch for ch in decomposed if ch in TONE_MARKS}
    stripped = "".join(ch for ch in decomposed if ch not in TONE_MARKS)
    vowel_positions = [pos for pos, ch in enumerate(stripped) if ch in "aeiouyAEIOUY"]
    options = [mark for mark in TONE_MARKS if mark not in present]
    if not vowel_positions or not options:
        return syllable
    insert_at = vowel_positions[-1] + 1
    while insert_at < len(stripped) and unicodedata.combining(stripped[insert_at]):
        insert_at += 1
    mark = options[rng.randrange(len(options))]
    candidate = unicodedata.normalize("NFC", stripped[:insert_at] + mark + stripped[insert_at:])
    if strip_diacritics(candidate) != strip_diacritics(syllable):
        return syllable
    return candidate


def fake_correct(text):
    """FAKE corrector for pipeline tests. It looks at the label of the current line."""
    syllables = normalize_nfc(text).split()
    label_syllables = _fake_state["gt"].split() if _fake_state["gt"] else []
    use_label = len(label_syllables) == len(syllables)
    out, conf = [], []
    for position, syllable in enumerate(syllables):
        rng = random.Random(f"fake|{syllable}|{position}")
        chosen, confidence = syllable, rng.uniform(0.3, 1.0)
        if use_label:
            target = label_syllables[position]
            if target != syllable and strip_diacritics(target) == strip_diacritics(syllable):
                if rng.random() < 0.7:  # a fixable error: fix it most of the time
                    chosen, confidence = target, rng.uniform(0.5, 1.0)
                else:
                    confidence = rng.uniform(0.0, 0.6)
            elif target == syllable:
                if rng.random() < 0.1:  # an already right syllable: break it sometimes
                    chosen, confidence = change_tone(syllable, rng), rng.uniform(0.0, 0.6)
                else:
                    confidence = rng.uniform(0.6, 1.0)
        out.append(chosen)
        conf.append(round(confidence, 4))
    return {"text": " ".join(out), "conf": conf}


if USE_FAKE:
    CORRECT_FN = fake_correct
else:
    assert REAL_CORRECT_FN is not None, "set REAL_CORRECT_FN in cell 2, or set USE_FAKE = True"
    CORRECT_FN = REAL_CORRECT_FN

RESULT_SUFFIX = ("_DRAFT" if OCR_IS_DRAFT else "") + ("_FAKE" if USE_FAKE else "")


def banner():
    if OCR_IS_DRAFT:
        print("*** DRAFT OCR: numbers below only check the pipeline ***")
    if USE_FAKE:
        print("*** FAKE corrector: numbers below only check the pipeline ***")


def correct_lines(lines):
    for line in lines:
        text_in = line["ocr_norm"]
        if USE_FAKE:
            _fake_state["gt"] = line["gt"]
        out = CORRECT_FN(text_in)
        problems = contract_checks.check_correct_output(text_in, out)
        assert not problems, f"{line['id']}: {problems[:3]}"
        line["corr_syll"] = out["text"].split()
        line["conf"] = list(out["conf"])
    _fake_state["gt"] = None


banner()
correct_lines(val_lines)
print("corrector output passes check_correct_output on all %d val lines" % len(val_lines))

## 7. Fixed / broken table, counted two ways

For every syllable we compare three things: the label, the OCR output and the corrected output.

| | corrected is right | corrected is wrong |
|---|---|---|
| **OCR was right** | kept right | **broken** |
| **OCR was wrong** | **fixed** | still wrong |

Broken rate = broken / (kept right + broken). Net fixed = fixed - broken.

Syllables can only be compared position by position when the label and the OCR line have the same number of syllables. Two ways to handle the other lines:
- **Option 1:** use only the lines with the same syllable count (clean, but drops lines).
- **Option 2:** align label and OCR syllables with `align.py` and use the matched or substituted pairs (uses most syllables).

In [ ]:
def prepare_pairs(lines):
    """Store the syllable pairs of both options on every line."""
    for line in lines:
        gt = line["gt"].split()
        ocr = line["ocr_norm"].split()
        line["pairs1"] = [(pos, pos) for pos in range(len(gt))] if len(gt) == len(ocr) else []
        pairs2 = []
        ref_pos = hyp_pos = 0
        for kind, _, _ in align(gt, ocr):
            if kind in (MATCH, SUB):
                pairs2.append((ref_pos, hyp_pos))
                ref_pos += 1
                hyp_pos += 1
            elif kind == INS:
                hyp_pos += 1
            else:
                ref_pos += 1
        line["pairs2"] = pairs2


def final_syllables(line, tau):
    ocr = line["ocr_norm"].split()
    if tau is None:
        return ocr
    conf = line["conf"]
    corr = line["corr_syll"]
    return [corr[pos] if conf[pos] >= tau else ocr[pos] for pos in range(len(ocr))]


def final_text(line, tau):
    return " ".join(final_syllables(line, tau))


def four_cell(lines, option, tau):
    counts = {"kept_right": 0, "broken": 0, "fixed": 0, "still_wrong": 0}
    used_lines = 0
    for line in lines:
        pairs = line["pairs1"] if option == 1 else line["pairs2"]
        if not pairs:
            continue
        used_lines += 1
        gt = line["gt"].split()
        ocr = line["ocr_norm"].split()
        final = final_syllables(line, tau)
        for ref_pos, hyp_pos in pairs:
            ocr_right = ocr[hyp_pos] == gt[ref_pos]
            final_right = final[hyp_pos] == gt[ref_pos]
            if ocr_right and final_right:
                counts["kept_right"] += 1
            elif ocr_right:
                counts["broken"] += 1
            elif final_right:
                counts["fixed"] += 1
            else:
                counts["still_wrong"] += 1
    right = counts["kept_right"] + counts["broken"]
    counts["syllables"] = right + counts["fixed"] + counts["still_wrong"]
    counts["lines_used"] = used_lines
    counts["broken_rate"] = counts["broken"] / right if right else 0.0
    counts["net_fixed"] = counts["fixed"] - counts["broken"]
    return counts


prepare_pairs(val_lines)
banner()
total_ref_syllables = sum(len(line["gt"].split()) for line in val_lines)
table_tau0 = {1: four_cell(val_lines, 1, 0.0), 2: four_cell(val_lines, 2, 0.0)}

print("correct everything (tau = 0.00):")
print("%-26s %14s %14s" % ("", "Option 1", "Option 2"))
rows = [
    ("lines used", "lines_used"), ("syllables compared", "syllables"),
    ("kept right", "kept_right"), ("broken", "broken"), ("fixed", "fixed"),
    ("still wrong", "still_wrong"), ("net fixed", "net_fixed"),
]
for title, key in rows:
    print("%-26s %14d %14d" % (title, table_tau0[1][key], table_tau0[2][key]))
print("%-26s %13.2f%% %13.2f%%" % ("broken rate", 100 * table_tau0[1]["broken_rate"], 100 * table_tau0[2]["broken_rate"]))
print("share of lines dropped by Option 1: %.1f%%" % (100.0 * (1 - table_tau0[1]["lines_used"] / len(val_lines))))
print("share of label syllables used: Option 1 %.1f%%, Option 2 %.1f%%" % (
    100.0 * table_tau0[1]["syllables"] / total_ref_syllables,
    100.0 * table_tau0[2]["syllables"] / total_ref_syllables))

## 8. Choose tau on val

The corrector's suggestion for a syllable is accepted only when its confidence is at least tau. The sweep covers `TAU_GRID` plus a "no correction" row. For each cap in `CAPS` the rule is: among the settings whose broken rate (Option 2) is at most the cap, take the one with the lowest CER; ties go to the higher tau. If "no correction" wins, the conclusion for that cap is not to correct.

Compare both caps below, then set `FINAL_CAP` in cell 2 by hand and run this cell again to lock tau.

In [ ]:
def sweep(lines):
    rows = []
    for tau in TAU_GRID + [None]:
        pairs = [(line["gt"], final_text(line, tau)) for line in lines]
        cer, wer = cer_wer(pairs)
        cell1 = four_cell(lines, 1, tau)
        cell2 = four_cell(lines, 2, tau)
        rows.append({
            "tau": tau, "cer": cer, "wer": wer,
            "broken_rate_opt2": cell2["broken_rate"], "fixed_opt2": cell2["fixed"], "broken_opt2": cell2["broken"],
            "net_fixed_opt2": cell2["net_fixed"], "broken_rate_opt1": cell1["broken_rate"],
            "net_fixed_opt1": cell1["net_fixed"],
        })
    return rows


def choose_tau(rows, cap):
    candidates = [row for row in rows if row["broken_rate_opt2"] <= cap]
    return min(candidates, key=lambda row: (row["cer"], -(row["tau"] if row["tau"] is not None else 2.0)))


def tau_label(tau):
    return "none" if tau is None else "%.2f" % tau


banner()
sweep_rows = sweep(val_lines)
print("%-6s %8s %8s %13s %11s %11s %13s" % (
    "tau", "CER", "WER", "broken opt2", "net opt2", "broken opt1", "net opt1"))
for row in sweep_rows:
    print("%-6s %7.2f%% %7.2f%% %12.2f%% %11d %10.2f%% %13d" % (
        tau_label(row["tau"]), 100 * row["cer"], 100 * row["wer"], 100 * row["broken_rate_opt2"],
        row["net_fixed_opt2"], 100 * row["broken_rate_opt1"], row["net_fixed_opt1"]))

chosen_by_cap = {}
print("\ncompare the caps:")
print("%-8s %-6s %8s %8s %13s %13s" % ("cap", "tau", "CER", "WER", "broken opt2", "broken opt1"))
for cap in CAPS:
    best = choose_tau(sweep_rows, cap)
    chosen_by_cap[cap] = best
    flag = "  (opt1 breaks the cap)" if best["broken_rate_opt1"] > cap else ""
    print("%-8s %-6s %7.2f%% %7.2f%% %12.2f%% %12.2f%%%s" % (
        "%.0f%%" % (100 * cap), tau_label(best["tau"]), 100 * best["cer"], 100 * best["wer"],
        100 * best["broken_rate_opt2"], 100 * best["broken_rate_opt1"], flag))
    if best["tau"] is None:
        print("   -> at this cap no correction beats correcting: the conclusion is not to correct")

import matplotlib.pyplot as plt

grid_rows = [row for row in sweep_rows if row["tau"] is not None]
figure, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot([row["tau"] for row in grid_rows], [100 * row["cer"] for row in grid_rows], marker="o")
axes[0].set_xlabel("tau")
axes[0].set_ylabel("CER on val (%)")
axes[0].set_title("CER by tau")
axes[1].plot([row["tau"] for row in grid_rows], [100 * row["broken_rate_opt2"] for row in grid_rows], marker="o")
for cap in CAPS:
    axes[1].axhline(100 * cap, linestyle="--", color="gray")
axes[1].set_xlabel("tau")
axes[1].set_ylabel("broken rate, Option 2 (%)")
axes[1].set_title("Broken rate by tau (dashed: caps)")
plt.tight_layout()
plt.show()

locked = None
if FINAL_CAP is None:
    print("FINAL_CAP is not set. Compare the caps above, set FINAL_CAP in cell 2, then run this cell again.")
else:
    assert FINAL_CAP in CAPS, "FINAL_CAP must be one of CAPS"
    best = choose_tau(sweep_rows, FINAL_CAP)
    locked = {
        "cap": FINAL_CAP, "tau": best["tau"], "val_cer": best["cer"], "val_wer": best["wer"],
        "basis": "val, Option 2 broken rate", "suffix": RESULT_SUFFIX,
    }
    write_json(f"{OUT_DIR}/eval_tau{RESULT_SUFFIX}.json", locked)

## 9. Bootstrap

Lines of one passage are not independent (many writers copy the same text), so the resampling unit is the **passage**: draw as many passages as there are, with replacement, pool their lines, compute CER before and after, and repeat `N_BOOT` times. The interval is the 2.5% to 97.5% range of the CER drop (before minus after). If the interval contains 0, do not claim an improvement.

The page-level table is a comparison only (val has 30 pages, but pages of one passage share the text). On test the two levels almost coincide.

In [ ]:
def bootstrap(lines, tau, unit):
    by_unit = collections.defaultdict(lambda: [0, 0, 0])
    for line in lines:
        key = line["passage"] if unit == "passage" else line["page"]
        by_unit[key][0] += edits(line["gt"], line["ocr_norm"])
        by_unit[key][1] += edits(line["gt"], final_text(line, tau))
        by_unit[key][2] += len(line["gt"])
    units = sorted(by_unit)
    total_before = sum(by_unit[key][0] for key in units)
    total_after = sum(by_unit[key][1] for key in units)
    total_chars = sum(by_unit[key][2] for key in units)
    rng = random.Random(SEED)
    drops = []
    for _ in range(N_BOOT):
        drawn = rng.choices(units, k=len(units))
        before = sum(by_unit[key][0] for key in drawn)
        after = sum(by_unit[key][1] for key in drawn)
        chars = sum(by_unit[key][2] for key in drawn)
        drops.append((before - after) / chars)
    drops.sort()
    return {
        "unit": unit, "units": len(units),
        "cer_before": total_before / total_chars, "cer_after": total_after / total_chars,
        "drop": (total_before - total_after) / total_chars,
        "ci_low": drops[int(0.025 * N_BOOT)], "ci_high": drops[int(0.975 * N_BOOT) - 1],
        "share_positive": sum(1 for value in drops if value > 0) / N_BOOT,
    }


def boot_configs():
    configs = [("tau=0.00 (correct all)", 0.0)]
    for cap in CAPS:
        configs.append(("cap %.0f%% -> tau %s" % (100 * cap, tau_label(chosen_by_cap[cap]["tau"])), chosen_by_cap[cap]["tau"]))
    return configs


def print_boot(results):
    print("%-26s %6s %9s %9s %9s %20s %8s" % ("setting", "units", "CER before", "CER after", "drop", "95% interval of drop", "drop>0"))
    for name, result in results:
        print("%-26s %6d %8.2f%% %8.2f%% %8.2f%% %9.2f%% to %6.2f%% %7.0f%%" % (
            name, result["units"], 100 * result["cer_before"], 100 * result["cer_after"], 100 * result["drop"],
            100 * result["ci_low"], 100 * result["ci_high"], 100 * result["share_positive"]))


banner()
boot_passage = [(name, bootstrap(val_lines, tau, "passage")) for name, tau in boot_configs()]
boot_page = [(name, bootstrap(val_lines, tau, "page")) for name, tau in boot_configs()]
print("by passage (main):")
print_boot(boot_passage)
print("\nby page (comparison, val only):")
print_boot(boot_page)

## 10. Score the test split (once)

Off by default. It runs only when all of these hold: `RUN_TEST = True`, the OCR file is final and the corrector is the real one (no `_DRAFT` or `_FAKE` suffix), `eval_tau.json` exists, and `eval_test_result.json` does not exist yet. After the run the result file blocks any second run. The OCR file must contain all test lines of `splits.json`.

In [ ]:
TAU_PATH = f"{OUT_DIR}/eval_tau.json"
TEST_RESULT_PATH = f"{OUT_DIR}/eval_test_result.json"


def run_test():
    if not RUN_TEST:
        print("RUN_TEST is False: the test split is not scored.")
        return
    if RESULT_SUFFIX:
        print("refused: the output is marked %s. Use the final OCR file and the real corrector first." % RESULT_SUFFIX)
        return
    if not os.path.exists(TAU_PATH):
        print("refused: lock tau on val first (eval_tau.json is missing).")
        return
    if os.path.exists(TEST_RESULT_PATH):
        print("refused: the test split was already scored once (eval_test_result.json exists).")
        return
    tau = read_json(TAU_PATH)["tau"]
    test_lines = build_lines(read_jsonl(OCR_OUT_PATH), "test")
    correct_lines(test_lines)
    prepare_pairs(test_lines)
    pairs_before_raw = [(line["gt"], line["ocr_raw"]) for line in test_lines]
    pairs_before_norm = [(line["gt"], line["ocr_norm"]) for line in test_lines]
    pairs_after = [(line["gt"], final_text(line, tau)) for line in test_lines]
    result = {
        "tau": tau,
        "lines": len(test_lines),
        "cer_wer_before_raw": cer_wer(pairs_before_raw),
        "cer_wer_before_normalized": cer_wer(pairs_before_norm),
        "cer_wer_after": cer_wer(pairs_after),
        "table_option1": four_cell(test_lines, 1, tau),
        "table_option2": four_cell(test_lines, 2, tau),
        "bootstrap_passage": bootstrap(test_lines, tau, "passage"),
    }
    for key, value in result.items():
        print(key, value)
    write_json(TEST_RESULT_PATH, result)


run_test()

## 11. Save the val results to Drive

In [ ]:
summary = {
    "suffix": RESULT_SUFFIX,
    "ocr_out_path": OCR_OUT_PATH,
    "val_lines": len(val_lines),
    "settings": {
        "caps": CAPS, "final_cap": FINAL_CAP, "n_boot": N_BOOT, "seed": SEED,
        "p_min": P_MIN, "ceiling_min": CEILING_MIN, "use_fake": USE_FAKE, "ocr_is_draft": OCR_IS_DRAFT,
    },
    "ocr_quality": {
        "cer_raw": cer_raw, "wer_raw": wer_raw, "cer_normalized": cer_norm, "wer_normalized": wer_norm,
        "empty_lines": empty_lines, "same_syllable_count_lines": same_count,
    },
    "gate": gate_report,
    "table_tau0": {"option1": table_tau0[1], "option2": table_tau0[2]},
    "sweep": sweep_rows,
    "chosen_by_cap": {str(cap): row for cap, row in chosen_by_cap.items()},
    "locked": locked,
    "bootstrap_passage": [{"setting": name, **result} for name, result in boot_passage],
    "bootstrap_page": [{"setting": name, **result} for name, result in boot_page],
}
banner()
write_json(f"{OUT_DIR}/eval_val{RESULT_SUFFIX}.json", summary)